In [2]:
#install dependancy

! pip install -q groq ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 86.2 MB/s eta 0:00:00


In [3]:
# API key integrate
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your API key: ")

print("API key set." if os.environ.get("GROQ_API_KEY") else "No API key found")

Enter your API key: ··········
API key set.


In [4]:
from groq import Groq
client=Groq()
for m in client.models.list().data:
  print(m.id)

allam-2-7b
canopylabs/orpheus-v1-english
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
qwen/qwen3.8-27b
openai/gpt-oss-120b
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-safeguard-20b
whisper-large-v3-turbo


In [5]:
from groq import Groq
from dataclasses import dataclass, field
from typing import List,Dict
@dataclass
class Chatbot:
  model:str="openai/gpt-oss-120b"
  system_prompt:str="You are a helpful,concise assistant."
  temperature: float=0.7
  max_tokens: int=1024
  _client: Groq=field(default=None,init=False)
  history: list[Dict[str,str]]=field(default_factory=list,repr=False)
  def __post_init__(self):
    self._client=Groq()

  def send(self,user_message:str)-> str:
    self.history.append({"role":"user","content":user_message}) # Add user message to history
    messages=[{"role":"user","content":self.system_prompt},*self.history] # Include system prompt and history
    try:
      response=self._client.chat.completions.create(
          model=self.model,
          messages=messages,
          temperature=self.temperature,
          max_tokens=self.max_tokens
      )
    except Exception as e:
        self.history.pop() #remove errors
        raise RuntimeError(f"Groq API error: {e}") from e

    #reply to user
    reply=response.choices[0].message.content
    self.history.append({"role":"assistant","content":reply}) # Add assistant reply to history
    return reply

  def reset(self):
    self.history.clear()

In [6]:
bot= Chatbot()
bot_msg=bot.send("You are a genius explainer.I want knowledge of each continent.")
print(bot_msg)

**Continents at a Glance**

| Continent | Area (≈ million km²) | Population (≈ 2024) | Number of Countries* | Notable Features |
|-----------|-------------------|--------------------|----------------------|------------------|
| **Asia** | 44.6 | 4.8 billion | 49 | Largest, most populous; home to the Himalayas, Gobi Desert, and the world’s two most‑populated countries (China, India). |
| **Africa** | 30.4 | 1.4 billion | 55 | Second‑largest; rich in biodiversity (Sahara, Congo Basin, Serengeti) and cultural diversity; fastest‑growing population. |
| **North America** | 24.7 | 600 million | 23 | Includes Canada, USA, Mexico; varied climates from Arctic tundra to tropical Caribbean; major economic hub. |
| **South America** | 17.8 | 435 million | 12 | Amazon rainforest (largest tropical forest), Andes mountains, Brazil (largest country by area in the region). |
| **Antarctica** | 14.0 | < 1,000 (research staff) | 0 (no sovereign states) | Ice‑covered continent; governed by the Antarctic T

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML
import html
import traceback

bot = Chatbot(model="openai/gpt-oss-20b")   # default

model_dropdown = widgets.Dropdown(
    options=["openai/gpt-oss-20b", "openai/gpt-oss-120b", "qwen/qwen3.8-27b", "allam-2-7b"],
    value=bot.model,
    description="Model:",
    layout=widgets.Layout(width="300px"),
)

temp_slider = widgets.FloatSlider(
    value=bot.temperature, min=0.0, max=1.0, step=0.1,
    description="Temp:", continuous_update=False,
    layout=widgets.Layout(width="300px"),
)

chat_log = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc", height="400px", overflow_y="auto", padding="8px")
)
text_input = widgets.Text(placeholder="Type a message and press Enter...", layout=widgets.Layout(width="80%"))
send_button = widgets.Button(description="Send", button_style="primary")
clear_button = widgets.Button(description="Clear chat", button_style="warning")
status_label = widgets.Label(value="")

input_row = widgets.HBox([text_input, send_button, clear_button])
controls_row = widgets.HBox([model_dropdown, temp_slider])
ui = widgets.VBox([controls_row, chat_log, input_row, status_label])

def render_bubble(role, text):
    if role == "user":
        bg, color, align = "#DCF8C6", "#000000", "right"
    elif role == "assistant":
        bg, color, align = "#F1F0F0", "#000000", "left"
    else:  # error
        bg, color, align = "#FFD6D6", "#7A0000", "left"

    safe_text = html.escape(text).replace("\n", "<br>")
    bubble = f"""
    <div style="text-align:{align}; margin:6px 0;">
      <span style="display:inline-block; background:{bg}; color:{color}; padding:8px 12px;
                    border-radius:10px; max-width:75%; text-align:left;">
        <b>{role}:</b><br>{safe_text}
      </span>
    </div>
    """
    with chat_log:
        display(HTML(bubble))


def on_send(_=None):
    message = text_input.value.strip()
    if not message:
        return
    text_input.value = ""
    text_input.disabled = send_button.disabled = True
    status_label.value = "Waiting for response..."
    render_bubble("user", message)

    bot.model = model_dropdown.value
    bot.temperature = temp_slider.value

    try:
        reply = bot.send(message)
        render_bubble("assistant", reply)
        status_label.value = ""
    except Exception as e:
        render_bubble("error", str(e))
        status_label.value = "Error — see message above."
        traceback.print_exc()
    finally:
        text_input.disabled = send_button.disabled = False


def on_clear(_=None):
    bot.reset()
    chat_log.clear_output()
    status_label.value = "Chat cleared."


send_button.on_click(on_send)
clear_button.on_click(on_clear)
text_input.on_submit(on_send)

display(ui)

NameError: name 'Chatbot' is not defined

In [7]:
bot = Chatbot(model="openai/gpt-oss-20b")

In [8]:
from chatbot import Chatbot
import traceback
import ipywidgets as widgets

bot = Chatbot(model="openai/gpt-oss-20b")

model_dropdown = widgets.Dropdown(
    options=[
        "openai/gpt-oss-20b"
    ],
    value="openai/gpt-oss-20b",
    description="Model:"
)

ModuleNotFoundError: No module named 'chatbot'

In [9]:
Chatbot(model="openai/gpt-oss-20b")

Chatbot(model='openai/gpt-oss-20b', system_prompt='You are a helpful,concise assistant.', temperature=0.7, max_tokens=1024, _client=<groq.Groq object at 0x7f07b46dd940>)

In [10]:
%pip list | grep -i chatbot

In [11]:
%pip list | grep -Ei "groq|openai|huggingface|gradio"

gradio                                6.26.0
gradio_client                         2.6.1
groq                                  1.7.0
hf-gradio                             0.4.1
huggingface_hub                       1.29.0
openai                                2.54.0


In [12]:
bot = Chatbot(model="openai/gpt-oss-20b")

In [13]:
%pip list | grep -Ei "groq|openai|huggingface|gradio"

gradio                                6.26.0
gradio_client                         2.6.1
groq                                  1.7.0
hf-gradio                             0.4.1
huggingface_hub                       1.29.0
openai                                2.54.0


In [14]:
bot = Chatbot(model="openai/gpt-oss-20b")

In [15]:
import os
from groq import Groq

bot = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [18]:
response = bot.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Hello, tell me about mount everest."
        }
    ]
)

print(response.choices[0].message.content)

**Mount Everest** – the world’s tallest peak – is a fascinating natural wonder that captures the imagination of climbers, scientists, and travelers alike. Below is a quick‑look guide covering the most important facts and context.

| Topic | Key points |
|-------|------------|
| **Location** | Lies on the Nepal–India border in the Himalaya range, in the Mahalangur section. |
| **Height** | 8 848 m (29 029 ft) above sea level. (A 2020 Chinese‑Nepalese survey revised the official elevation to 8 848 .86 m.) |
| **Geology** | Formed by the collision of the Indian and Eurasian tectonic plates. The mountain is still rising at about 4 mm per year. |
| **First ascent** | 29 May 1953: Sir Edmund Hillary (New Zealand) & Tenzing Norgay (Tibetan‑Bhutanese Sherpa). |
| **Climbing routes** | *South Col* (from Nepal) – most popular, but still highly technical and dangerous.<br>*North Ridge* (from Tibet/China) – more technical and politically sensitive. |
| **Season** | Pre‑monsoon (April–May) and post